# Variables instrumentales y MC2E
### Econometría II
**Alejandro Mosiño** — Universidad de Guanajuato

In [ ]:
# Ejecutar solo la primera vez
# install.packages(c("ivreg", "car"))

## Datos y modelo MCO

Usamos datos de mujeres casadas que trabajan para estimar la ecuación de salarios:

$$
\ln(\text{wage}_i) = \beta_1 + \beta_2 \, \text{educ}_i + \beta_3 \, \text{exper}_i + \beta_4 \, \text{exper}_i^2 + u_i
$$

Si `educ` es endógena (por ejemplo, por habilidad no observada), el estimador MCO es inconsistente.

In [ ]:
library(ivreg)
library(car)

datos <- read.csv("E206--mroz.csv", sep = "\t")
datos <- subset(datos, wage > 0)

mod_mco <- lm(log(wage) ~ educ + exper + I(exper^2), data = datos)
summary(mod_mco)

## Estimación por VI

Usamos la educación de la madre (`mothereduc`) como instrumento de `educ`.

In [ ]:
mod_iv1 <- ivreg(log(wage) ~ educ + exper + I(exper^2) | mothereduc + exper + I(exper^2),
                 data = datos)
summary(mod_iv1)

## Estimación por MC2E

Con dos instrumentos (`mothereduc` y `fathereduc`) estimamos por mínimos cuadrados en dos etapas.

Paso a paso:
1. **Primera etapa:** regresamos la variable endógena sobre las exógenas y los instrumentos.
2. **Segunda etapa:** sustituimos `educ` por su valor ajustado en el modelo original.

In [ ]:
etapa1 <- lm(educ ~ mothereduc + fathereduc + exper + I(exper^2), data = datos)
datos$educ_hat <- fitted(etapa1)

etapa2 <- lm(log(wage) ~ educ_hat + exper + I(exper^2), data = datos)
summary(etapa2)

Los coeficientes son los de MC2E, pero los errores estándar de esta segunda etapa **no** son válidos. Con la librería `ivreg`:

In [ ]:
mod_iv2 <- ivreg(log(wage) ~ educ + exper + I(exper^2) | mothereduc + fathereduc + exper + I(exper^2),
                 data = datos)
summary(mod_iv2)

## Instrumentos débiles

La relevancia de los instrumentos se evalúa en la primera etapa: probamos que sean conjuntamente significativos.

$$
H_0: \pi_1 = \pi_2 = 0 \qquad H_1: \text{al menos un } \pi_j \neq 0
$$

Como regla práctica, un estadístico $F$ mayor a 10 indica instrumentos fuertes.

In [ ]:
summary(etapa1)
linearHypothesis(etapa1, c("fathereduc = 0", "mothereduc = 0"))

## Prueba de Sargan

$$
H_0: \text{todos los instrumentos son exógenos} \qquad H_1: \text{al menos un instrumento no lo es}
$$

Paso a paso: regresamos los residuales de MC2E sobre las variables exógenas y los instrumentos, y calculamos $S = n R^2$, que bajo $H_0$ se distribuye $\chi^2$ con tantos grados de libertad como instrumentos excedentes (aquí, 1).

In [ ]:
datos$e_iv <- residuals(mod_iv2)

aux_sargan <- lm(e_iv ~ mothereduc + fathereduc + exper + I(exper^2), data = datos)
S <- nobs(aux_sargan) * summary(aux_sargan)$r.squared

qchisq(0.05, df = 1, lower.tail = FALSE)
pchisq(S, df = 1, lower.tail = FALSE)

Con la librería `ivreg`, que reporta también la prueba de instrumentos débiles y la de Wu-Hausman:

In [ ]:
summary(mod_iv2, diagnostics = TRUE)

## Prueba de Hausman

Probamos si `educ` es realmente endógena, es decir, si MCO y MC2E difieren de forma sistemática.

$$
H_0: \text{Cov}(\text{educ}_i, u_i) = 0 \qquad H_1: \text{Cov}(\text{educ}_i, u_i) \neq 0
$$

Paso a paso: tomamos los residuales de la primera etapa, $\hat{v}_i$, y los agregamos como regresor al modelo original. Bajo $H_0$, su coeficiente $\delta$ es igual a cero (prueba $t$).

In [ ]:
datos$v_hat <- residuals(etapa1)

aux_hausman <- lm(log(wage) ~ educ + exper + I(exper^2) + v_hat, data = datos)
summary(aux_hausman)

Este resultado es equivalente a la prueba de Wu-Hausman que reporta `summary(mod_iv2, diagnostics = TRUE)` (el estadístico $F$ es el cuadrado del estadístico $t$ de `v_hat`).